# Multi-Agent LLM Research Automation Platform
## Full Research Pipeline - Paper Generation

This notebook runs the complete multi-agent research pipeline to generate a research paper.

**Workflow**: Topic Discovery → Domain Analysis → Literature Review → Gap Analysis → Report Generation

---

## 1. Setup & Configuration

In [32]:
# Setup Python path
import sys
import os
from pathlib import Path

# Get project root (parent of notebooks folder)
NOTEBOOK_DIR = Path('__file__').parent if '__file__' in dir() else Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
sys.path.insert(0, str(PROJECT_ROOT))

# Load environment variables
from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / '.env')

print("Setup complete!")
print(f"Project Root: {PROJECT_ROOT}")
print(f"LLM Status: {os.getenv('LLM_STATUS', 'OFFLINE')}")

Setup complete!
Project Root: d:\SEM 6\AIML317_PROJECT_III\project_sgp
LLM Status: OFFLINE


## 2. Define Research Topic

Change the `RESEARCH_TOPIC` below to your desired research topic.

In [33]:
# Research Configuration
RESEARCH_TOPIC = "Transformer Models in Medical Image Analysis"  # CHANGE THIS
JOB_ID = "notebook_research_001"

print(f"Research Topic: {RESEARCH_TOPIC}")
print(f"Job ID: {JOB_ID}")

Research Topic: Transformer Models in Medical Image Analysis
Job ID: notebook_research_001


## 3. Load Agents

In [34]:
# Load agent registry
from ai_engine.agents.registry import AGENTS

print(f"Loaded {len(AGENTS)} agents")

Loaded 38 agents


## 4. Run Full Research Pipeline

In [35]:
import time

# Initialize state
state = {
    "task": RESEARCH_TOPIC,
    "_job_id": JOB_ID,
    "findings": {},
    "history": [],
    "topic_locked": False,
    "selected_topic": None,
    "topic_suggestions": []
}

print("=" * 60)
print("STARTING FULL RESEARCH PIPELINE")
print("=" * 60)

start_time = time.time()

STARTING FULL RESEARCH PIPELINE


In [36]:
# Step 1: Topic Discovery
print("\n[STEP 1/5] Topic Discovery...")
try:
    result = AGENTS['topic_discovery'].run(state)
    if result.get('topic_locked'):
        state['topic_locked'] = True
        state['selected_topic'] = result.get('selected_topic')
        print(f"    Topic locked: {state['selected_topic'][:50]}...")
    else:
        state['topic_suggestions'] = result.get('topic_suggestions', [])
        print(f"    Generated {len(state['topic_suggestions'])} suggestions")
        if state['topic_suggestions']:
            state['selected_topic'] = state['topic_suggestions'][0].get('title', RESEARCH_TOPIC)
            state['topic_locked'] = True
except Exception as e:
    print(f"    Error: {e}")
    state['selected_topic'] = RESEARCH_TOPIC
    state['topic_locked'] = True

state['task'] = state['selected_topic']


[STEP 1/5] Topic Discovery...
    Error: 'NoneType' object has no attribute 'run'


In [37]:
# Step 2: Domain Intelligence
print("\n[STEP 2/5] Domain Intelligence...")
try:
    result = AGENTS['domain_intelligence'].run(state)
    state['findings']['domain_intelligence'] = result
    state['history'].append("domain_intelligence completed")
    print("    Domain analysis complete")
except Exception as e:
    print(f"    Error: {e}")


[STEP 2/5] Domain Intelligence...
    Error: 'NoneType' object has no attribute 'run'


In [38]:
# Step 3: Literature Review
print("\n[STEP 3/5] Literature Review...")
try:
    result = AGENTS['slr'].run(state)
    state['findings']['slr'] = result
    state['history'].append("slr completed")
    print("    Literature review complete")
except Exception as e:
    print(f"    Error: {e}")


[STEP 3/5] Literature Review...
    Error: 'NoneType' object has no attribute 'run'


In [39]:
# Step 4: Gap Synthesis
print("\n[STEP 4/5] Gap Synthesis...")
try:
    result = AGENTS['gap_synthesis'].run(state)
    state['findings']['gap_synthesis'] = result
    state['history'].append("gap_synthesis completed")
    print("    Gap analysis complete")
except Exception as e:
    print(f"    Error: {e}")


[STEP 4/5] Gap Synthesis...
    Error: 'NoneType' object has no attribute 'run'


In [40]:
# Step 5: Scientific Writing
print("\n[STEP 5/5] Scientific Writing...")
try:
    result = AGENTS['scientific_writing'].run(state)
    state['findings']['scientific_writing'] = result
    state['history'].append("scientific_writing completed")
    report_data = result.get('response', {})
    md_report = report_data.get('markdown_report', '') if isinstance(report_data, dict) else ''
    print(f"    Report generated: {len(md_report)} characters")
except Exception as e:
    print(f"    Error: {e}")
    md_report = ''

elapsed = time.time() - start_time
print("\n" + "=" * 60)
print(f"PIPELINE COMPLETED in {elapsed:.1f} seconds")
print("=" * 60)


[STEP 5/5] Scientific Writing...
    Error: 'NoneType' object has no attribute 'run'

PIPELINE COMPLETED in 0.6 seconds


## 5. View Generated Report

In [41]:
# Get the generated report
report_data = state.get('findings', {}).get('scientific_writing', {}).get('response', {})
markdown_report = report_data.get('markdown_report', '') if isinstance(report_data, dict) else ''

if markdown_report:
    print("GENERATED RESEARCH REPORT")
    print("=" * 60)
    print(markdown_report)
else:
    print("No report generated. Check errors above.")

No report generated. Check errors above.


## 6. Save Output Files

In [42]:
# Create output directory
output_dir = PROJECT_ROOT / 'output' / JOB_ID
output_dir.mkdir(parents=True, exist_ok=True)

# Save report
if markdown_report:
    md_path = output_dir / 'research_report.md'
    with open(md_path, 'w', encoding='utf-8') as f:
        f.write(markdown_report)
    print(f"Report saved: {md_path}")

# Save state summary
import json
state_path = output_dir / 'pipeline_state.json'
state_summary = {
    'topic': state.get('selected_topic'),
    'history': state.get('history', []),
    'findings_keys': list(state.get('findings', {}).keys()),
    'execution_time': elapsed
}
with open(state_path, 'w', encoding='utf-8') as f:
    json.dump(state_summary, f, indent=2)
print(f"State saved: {state_path}")
print(f"\nOutputs in: {output_dir}")

State saved: d:\SEM 6\AIML317_PROJECT_III\project_sgp\output\notebook_research_001\pipeline_state.json

Outputs in: d:\SEM 6\AIML317_PROJECT_III\project_sgp\output\notebook_research_001


## 7. Pipeline Summary

In [43]:
print("PIPELINE EXECUTION SUMMARY")
print("=" * 60)
print(f"Topic: {state.get('selected_topic', RESEARCH_TOPIC)}")
print(f"Execution Time: {elapsed:.1f} seconds")
print()
print("Steps Completed:")
for i, step in enumerate(state.get('history', []), 1):
    print(f"  {i}. {step}")
print()
print("Findings:")
for key in state.get('findings', {}).keys():
    print(f"  - {key}")
print()
if markdown_report:
    print(f"Report: {len(markdown_report)} chars, {markdown_report.count('# ')} sections")

PIPELINE EXECUTION SUMMARY
Topic: Transformer Models in Medical Image Analysis
Execution Time: 0.6 seconds

Steps Completed:

Findings:



---
## Complete!

**Files:**
- `output/{JOB_ID}/research_report.md`
- `output/{JOB_ID}/pipeline_state.json`

**Next Steps:** Review, edit, and add citations to the report.